## PhytoScan: AI-Assisted Plant Disease Diagnosis and Guidance

Trains a plant-disease image classifier (MobileNetV2 transfer learning on PlantVillage) and exports it for a TensorFlow.js web app.

**Run top to bottom, once.** Tip: switch to a GPU runtime (Runtime > Change runtime type > T4 GPU) before running, training on CPU is very slow.

### 1. Setup

In [ ]:
import os, json, pathlib, random, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

# ---- Config ----
SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_STAGE1 = 5      # frozen backbone
EPOCHS_STAGE2 = 5      # fine-tuning
FINE_TUNE_LAYERS = 30  # how many top backbone layers to unfreeze
MODEL_PATH = "plant_disease_mobilenetv2.keras"
LABELS_PATH = "plant_disease_labels.json"

keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU available:", gpus[0].name)
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("Running on CPU. Switch Runtime to T4 GPU for much faster training.")

### 2. Download the dataset

Uses `kagglehub`, which downloads and caches the dataset without any interactive prompts (no more `replace ...? [y]es, [n]o` freezes from `unzip`).
If Kaggle asks for credentials, add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's **Secrets** panel (key icon on the left) and re-run.

In [ ]:
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

download_path = pathlib.Path(kagglehub.dataset_download("abdallahalidev/plantvillage-dataset"))
print("Downloaded to:", download_path)

### 2.1 Locate the **color** images and list classes

The Kaggle zip contains three versions of the data (`color`, `grayscale`, `segmented`). The old code grabbed whichever folder it found first, which was `grayscale`. This version explicitly prefers `color`.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png"}

def find_dataset_root(base):
    candidates = []
    for dirpath, dirnames, _ in os.walk(base):
        if len(dirnames) >= 30 and sum("___" in d for d in dirnames) >= 30:
            candidates.append(pathlib.Path(dirpath))
    if not candidates:
        raise FileNotFoundError(f"No PlantVillage class folders found under {base}")
    for c in candidates:
        if c.name.lower() == "color":
            return c
    preferred = [c for c in candidates if c.name.lower() not in ("grayscale", "segmented")]
    return (preferred or candidates)[0]

dataset_root = find_dataset_root(download_path)
print("Dataset root:", dataset_root)

class_names = sorted(d.name for d in dataset_root.iterdir() if d.is_dir())
num_classes = len(class_names)
label_to_index = {n: i for i, n in enumerate(class_names)}

all_image_paths, all_image_labels = [], []
for name in class_names:
    files = sorted(p for p in (dataset_root / name).iterdir() if p.suffix.lower() in IMG_EXTS)
    all_image_paths += [str(p) for p in files]
    all_image_labels += [label_to_index[name]] * len(files)

all_image_paths = np.array(all_image_paths)
all_image_labels = np.array(all_image_labels, dtype=np.int32)
assert num_classes > 0 and len(all_image_paths) == len(all_image_labels) > 0
print(f"Found {num_classes} classes and {len(all_image_paths)} images.")

### 2.2 Dataset exploration

In [ ]:
counts = np.bincount(all_image_labels, minlength=num_classes)
print(f"Total images: {len(all_image_paths)} | Classes: {num_classes}\n")
for i, (n, c) in enumerate(zip(class_names, counts)):
    print(f"{i:2d}: {n:55s} {c}")

plt.figure(figsize=(8, 10))
plt.barh(range(num_classes), counts)
plt.yticks(range(num_classes), class_names, fontsize=7)
plt.xlabel("Images")
plt.title("Images per class")
plt.tight_layout()
plt.show()

### 2.3 Sample images

In [ ]:
rng_vis = np.random.default_rng(SEED)
idxs = rng_vis.choice(len(all_image_paths), 9, replace=False)
plt.figure(figsize=(10, 10))
for i, idx in enumerate(idxs):
    plt.subplot(3, 3, i + 1)
    plt.imshow(keras.utils.load_img(all_image_paths[idx], target_size=IMAGE_SIZE))
    plt.title(class_names[all_image_labels[idx]], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

### 2.4 Stratified 70 / 15 / 15 split (deterministic)

In [ ]:
rng = np.random.default_rng(SEED)
tr_i, va_i, te_i = [], [], []
for cid in range(num_classes):
    idx = np.where(all_image_labels == cid)[0]
    rng.shuffle(idx)
    n = len(idx)
    a, b = int(0.70 * n), int(0.85 * n)
    tr_i.append(idx[:a]); va_i.append(idx[a:b]); te_i.append(idx[b:])

tr_i, va_i, te_i = (np.concatenate(x) for x in (tr_i, va_i, te_i))
train_paths, train_labels = all_image_paths[tr_i], all_image_labels[tr_i]
val_paths,   val_labels   = all_image_paths[va_i], all_image_labels[va_i]
test_paths,  test_labels  = all_image_paths[te_i], all_image_labels[te_i]

print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")
assert len(set(train_paths) & set(test_paths)) == 0, "Data leakage between train and test!"

### 2.5 tf.data pipelines

Notes on what changed vs. the old notebook:
- **No `.cache()`** of float32 images (54k images x 600 KB would need about 30 GB of RAM and crash Colab).
- Augmentation and MobileNetV2 scaling now live **inside the model** (section 3). That means the exported TF.js model accepts raw 0-255 pixels, so the web app doesn't need to re-implement preprocessing.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image_and_label(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMAGE_SIZE)
    img.set_shape(IMAGE_SIZE + (3,))
    return img, label            # float32, values 0-255

def make_ds(paths, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    return (ds.map(load_image_and_label, num_parallel_calls=AUTOTUNE)
              .batch(BATCH_SIZE)
              .prefetch(AUTOTUNE))

train_ds = make_ds(train_paths, train_labels, training=True)
val_ds   = make_ds(val_paths,   val_labels,   training=False)
test_ds  = make_ds(test_paths,  test_labels,  training=False)   # no shuffle: order matches test_labels
print("Data pipelines ready.")

### 3. Build the MobileNetV2 model

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
], name="data_augmentation")          # only active during training

base_model = keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=IMAGE_SIZE + (3,), name="image")   # raw pixels 0-255
x = data_augmentation(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_preprocess")(x)   # same as preprocess_input
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
model = keras.Model(inputs, outputs, name="phytoscan_mobilenetv2")
model.summary()

### 4. Two-stage transfer learning

**Stage 1** trains only the new classification head. **Stage 2** unfreezes the top of MobileNetV2 and fine-tunes with a tiny learning rate (BatchNorm layers stay frozen, which keeps fine-tuning stable).

In [ ]:
def make_callbacks(path, min_lr):
    return [
        ModelCheckpoint(path, monitor="val_accuracy", mode="max", save_best_only=True),
        ReduceLROnPlateau(monitor="val_accuracy", mode="max", factor=0.2, patience=2, min_lr=min_lr, verbose=1),
        EarlyStopping(monitor="val_accuracy", mode="max", patience=4, restore_best_weights=True, verbose=1),
    ]

print("--- Stage 1: feature extraction ---")
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_stage1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE1,
                           callbacks=make_callbacks("best_model_stage1.keras", 1e-6))

In [ ]:
print("--- Stage 2: fine-tuning ---")
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),     # must recompile after changing trainable
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_stage2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE2,
                           callbacks=make_callbacks(MODEL_PATH, 1e-7))

### 5. Training curves

In [ ]:
def plot_history(h, title):
    acc, vacc = h.history["accuracy"], h.history["val_accuracy"]
    loss, vloss = h.history["loss"], h.history["val_loss"]
    ep = range(1, len(acc) + 1)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(ep, acc, "b-o", label="Train"); plt.plot(ep, vacc, "r-s", label="Val")
    plt.title(f"Accuracy - {title}"); plt.xlabel("Epoch"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(ep, loss, "b-o", label="Train"); plt.plot(ep, vloss, "r-s", label="Val")
    plt.title(f"Loss - {title}"); plt.xlabel("Epoch"); plt.legend()
    plt.tight_layout(); plt.show()

plot_history(history_stage1, "Stage 1: feature extraction")
plot_history(history_stage2, "Stage 2: fine-tuning")

### 6. Test evaluation (full held-out test set)

The earlier version only evaluated on 10 batches, so its numbers were not meaningful. This evaluates all test images.

In [ ]:
final_model = keras.models.load_model(MODEL_PATH)

test_loss, test_acc = final_model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(final_model.predict(test_ds, verbose=0), axis=1)

print("\nClassification report:")
print(classification_report(y_true, y_pred, labels=list(range(num_classes)),
                            target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(16, 13))
sns.heatmap(cm, cmap="Blues", xticklabels=class_names, yticklabels=class_names, annot=False)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix (full test set)")
plt.xticks(fontsize=6, rotation=90); plt.yticks(fontsize=6)
plt.tight_layout(); plt.show()

### 7. Top-3 predictions function

In [ ]:
def get_top_n_predictions(model, image_path, class_names, n=3):
    """Returns the top-n classes with probabilities. The model already outputs softmax
    probabilities and does its own preprocessing, so we feed raw 0-255 pixels."""
    img = keras.utils.load_img(image_path, target_size=IMAGE_SIZE)
    arr = np.expand_dims(keras.utils.img_to_array(img), 0)
    probs = model.predict(arr, verbose=0)[0]
    top = np.argsort(probs)[::-1][:n]
    return [{"class_name": class_names[i], "probability": float(probs[i])} for i in top]

sample_path, sample_label = test_paths[0], class_names[test_labels[0]]
print(f"True label: {sample_label}")
for p in get_top_n_predictions(final_model, sample_path, class_names):
    print(f"  {p['class_name']}: {p['probability']:.2%}")

### 8. Save class labels

In [ ]:
label_map = {str(i): name for i, name in enumerate(class_names)}
with open(LABELS_PATH, "w") as f:
    json.dump(label_map, f, indent=2)

assert num_classes == len(label_map)
print(f"Saved {len(label_map)} labels to {LABELS_PATH}")

### 9. Verify the saved model

In [ ]:
reloaded = keras.models.load_model(MODEL_PATH)
batch, _ = next(iter(test_ds))
out_a = final_model.predict(batch, verbose=0)
out_b = reloaded.predict(batch, verbose=0)

assert out_b.shape == (batch.shape[0], num_classes), "Output shape mismatch"
assert np.allclose(out_a, out_b, atol=1e-5), "Reloaded model gives different predictions"
print("Saved model reloads correctly and reproduces predictions.")

### 10. Export for the web app (TensorFlow.js)

Optional. TF.js converter support for Keras 3 (`.keras`) files can be version-sensitive, so this is wrapped in a try/except and won't break the notebook if conversion fails.

In [ ]:
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflowjs"])
    os.makedirs("tfjs_model", exist_ok=True)
    subprocess.check_call(["tensorflowjs_converter", "--input_format=keras_keras", MODEL_PATH, "tfjs_model"])
    print("TF.js model written to ./tfjs_model")
except Exception as e:
    print("TF.js conversion skipped:", e)
    print("Fallback: run `pip install tensorflowjs` in a fresh runtime and convert plant_disease_mobilenetv2.keras locally.")

### 11. Final summary

In [ ]:
print("--- PhytoScan summary ---")
print(f"Classes:          {num_classes}")
print(f"Train/Val/Test:   {len(train_paths)} / {len(val_paths)} / {len(test_paths)}")
print(f"Test accuracy:    {test_acc:.4f}")
print(f"Model file:       {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")
print(f"Labels file:      {LABELS_PATH}")

try:
    from google.colab import files
    files.download(MODEL_PATH)
    files.download(LABELS_PATH)
except Exception:
    pass